In [1]:
!pip install wandb -q
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: grigoryanarpi8 (grigoryanarpi8-npua) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [2]:
import os
import zipfile
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import collections
from torchvision import transforms
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter
import collections
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler
import collections
from torchvision import models
import numpy as np
from sklearn.metrics import average_precision_score
from tqdm import tqdm
import shutil
import random
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from PIL import Image
from torchvision import transforms





In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
zip_path = "/content/drive/MyDrive/disease/train.zip.zip"
val_zip = "/content/drive/MyDrive/disease/val.zip.zip"
extract_path = "/content/data"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)
with zipfile.ZipFile(val_zip, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipped successfully!")


Unzipped successfully!


In [5]:
TRAIN_PATH = "/content/data/train"
VAL_PATH   = "/content/data/val"
SAVE_PATH = "/content/drive/MyDrive/disease/best_model2.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cpu


In [6]:
label_map = {
    'black rot': 0, 'mosaic virus': 1, 'rust': 2, 'scab': 3,
    'anthracnose': 4, 'black leaf streak': 5, 'bunchy top': 6,
    'downy mildew': 7, 'pepper blossom end rot': 8,
    'pepper powdery mildew': 9, 'alternaria leaf spot': 10,
    'early blight': 11, 'leaf spot': 12, 'powdery mildew': 13,
    'canker': 14, 'greening disease': 15, 'berry blotch': 16,
    'leaf rust': 17, 'gray leaf spot': 18, 'northern leaf blight': 19,
    'smut': 20, 'angular leaf spot': 21, 'bacterial wilt': 22,
    'sheath blight': 23, 'tar spot': 24, 'brown rot': 25,
    'leaf curl': 26, 'late blight': 27, 'brown spot': 28,
    'frog eye leaf spot': 29, 'mosaic': 30, 'bacterial leaf spot': 31,
    'leaf mold': 32, 'septoria leaf spot': 33,
    'bacterial leaf streak (black chaff)': 34, 'head scab': 35,
    'loose smut': 36, 'septoria blotch': 37, 'stem rust': 38,
    'stripe rust': 39,
}
label_to_name = {v: k for k, v in label_map.items()}
num_classes   = len(label_map)
print(f" label_map : {num_classes} classes")

 label_map : 40 classes


In [7]:
class PlantDataset(Dataset):
    def __init__(self, root_dir, label_map, transform=None):
        self.root_dir = root_dir
        self.label_map = label_map
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.classes = []

        for c in sorted(os.listdir(root_dir)):
            if c.startswith(".") or not os.path.isdir(os.path.join(root_dir, c)):
                continue

            disease = " ".join(c.split(" ")[1:])
            if disease not in label_map:
                continue

            label = label_map[disease]
            if disease not in self.classes:
                self.classes.append(disease)

            class_path = os.path.join(root_dir, c)
            for img in os.listdir(class_path):
                if img.lower().endswith((".jpg", ".jpeg", ".png")):
                    self.image_paths.append(os.path.join(class_path, img))
                    self.labels.append(label)

        print(f"{root_dir} → {len(self.image_paths)} images | {len(self.classes)} classes")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx: int):
        if len(self.image_paths) == 0:
            raise RuntimeError("Dataset is empty — check your root_dir and label_map")

        img_path = self.image_paths[idx % len(self.image_paths)]

        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"  [WARN] Corrupt image skipped: {img_path} ({e})")
            return self.__getitem__((idx + 1) % len(self.image_paths))

        label = self.labels[idx % len(self.image_paths)]

        if self.transform:
            img = self.transform(img)

        return img, label

    def get_sample_weights(self, sqrt_balance=True):
        counts = Counter(self.labels)
        weights = [1.0 / (counts[l] ** 0.5 if sqrt_balance else counts[l]) for l in self.labels]
        mean_w = sum(weights) / len(weights)
        return [w / mean_w for w in weights]

In [8]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.1,
        hue=0.01
    ),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

In [9]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce)
        loss = (1 - pt) ** self.gamma * ce
        return loss.mean()

criterion = FocalLoss(gamma=2.0)
print("FocalLoss ready")

FocalLoss ready


In [10]:
train_dataset = PlantDataset(TRAIN_PATH, label_map, train_transform)
val_dataset = PlantDataset(VAL_PATH, label_map, val_transform)

train_classes = set(train_dataset.labels)
val_classes = set(val_dataset.labels)
missing = train_classes - val_classes

if missing:
    label_to_name = {v: k for k, v in label_map.items()}

    def find_class_folder(root, disease_name):
        for folder in sorted(os.listdir(root)):
            suffix = " ".join(folder.split(" ")[1:])
            if suffix == disease_name:
                return os.path.join(root, folder)
        return None

    for label_idx in missing:
        disease_name = label_to_name.get(label_idx)
        if not disease_name:
            print(f"  [WARN] No name found for label {label_idx}, skipping")
            continue

        print(f" Missing in val: class {label_idx} → '{disease_name}'")

        train_class_path = find_class_folder(TRAIN_PATH, disease_name)
        if not train_class_path:
            print(f"  [ERROR] Folder not found in train for '{disease_name}'")
            continue

        folder_name = os.path.basename(train_class_path)
        val_class_path = os.path.join(VAL_PATH, folder_name)
        os.makedirs(val_class_path, exist_ok=True)

        images = [
            f for f in sorted(os.listdir(train_class_path))
            if f.lower().endswith((".jpg"))
        ]

        num_to_move = max(5, min(10, int(len(images) * 0.2)))

        images_to_move = random.sample(images, num_to_move)

        for img in images_to_move:
            shutil.move(
                os.path.join(train_class_path, img),
                os.path.join(val_class_path, img)
            )

        print(f" Moved {num_to_move}/{len(images)} images → {val_class_path}")

    train_dataset = PlantDataset(TRAIN_PATH, label_map, train_transform)
    val_dataset   = PlantDataset(VAL_PATH,   label_map, val_transform)

    still_missing = set(train_dataset.labels) - set(val_dataset.labels)
    if still_missing:
        print(f"  [WARN] Still missing after fix: {still_missing}")
    else:
        print(" All classes now present in both splits")

else:
    print("All train classes present in val — no fix needed")

print(" Train class distribution:")
for label, count in sorted(Counter(train_dataset.labels).items(), key=lambda x: x[1], reverse=True):
    print(f"  Class {label}: {count} images")

values = list(Counter(train_dataset.labels).values())
print(f"\n  Min: {min(values)} | Max: {max(values)} | Mean: {sum(values)/len(values):.1f}")



/content/data/train → 7783 images | 40 classes
/content/data/val → 390 images | 39 classes
 Missing in val: class 9 → 'pepper powdery mildew'
 Moved 5/26 images → /content/data/val/bell pepper powdery mildew
/content/data/train → 7778 images | 40 classes
/content/data/val → 395 images | 40 classes
 All classes now present in both splits
 Train class distribution:
  Class 13: 840 images
  Class 2: 681 images
  Class 7: 679 images
  Class 14: 330 images
  Class 39: 298 images
  Class 11: 289 images
  Class 0: 283 images
  Class 3: 276 images
  Class 35: 259 images
  Class 17: 231 images
  Class 1: 230 images
  Class 27: 220 images
  Class 25: 194 images
  Class 12: 182 images
  Class 29: 178 images
  Class 37: 172 images
  Class 36: 155 images
  Class 4: 153 images
  Class 20: 142 images
  Class 22: 129 images
  Class 10: 122 images
  Class 21: 122 images
  Class 26: 122 images
  Class 5: 107 images
  Class 6: 97 images
  Class 32: 96 images
  Class 8: 90 images
  Class 15: 90 images
  C

In [11]:
values = list(Counter(train_dataset.labels).values())
print(f"Min: {min(values)} | Max: {max(values)} | Imbalance: {max(values)/min(values):.1f}:1")

sample_weights = train_dataset.get_sample_weights(sqrt_balance=True)
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

pin = torch.cuda.is_available()
train_loader = DataLoader(train_dataset, batch_size=64, sampler=sampler,
                          num_workers=2, pin_memory=pin)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False,
                          num_workers=2, pin_memory=pin)
print("DataLoaders ready")

Min: 21 | Max: 840 | Imbalance: 40.0:1
DataLoaders ready


In [13]:
counter = collections.Counter(train_dataset.labels)
print("Most common:", counter.most_common(5))
print("Least common:", counter.most_common()[-5:])

values = list(counter.values())
print(f"Imbalance ratio: {max(values)/min(values):.1f}:1")

Most common: [(13, 840), (2, 681), (7, 679), (14, 330), (39, 298)]
Least common: [(31, 90), (33, 90), (34, 90), (38, 90), (9, 21)]
Imbalance ratio: 40.0:1


In [12]:
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

in_features = model.fc.in_features

model.fc = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(512, num_classes)
)

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Trainable: {trainable:,} / {total:,} params ({100*trainable/total:.1f}%)")
print(f"Output classes: {num_classes}")

wandb.init(
    project="plant-disease-classification",
    name="resnet50_focal_sqrt_cosine",
    config={
        "model": "resnet50",
        "batch_size": 64,
        "frozen_epochs": 10,
        "fine_tune_epochs": 30,
        "frozen_lr": 1e-3,
        "fine_tune_lr": 1e-4,
        "weight_decay": 1e-4,
        "scheduler": "CosineAnnealingLR",
        "loss": "FocalLoss(gamma=2.0)",
        "sampler": "WeightedRandomSampler(sqrt)",
        "dropout": 0.4,
        "augmentation": "RandomResizedCrop(0.85)+ColorJitter",
        "num_classes": num_classes
    }
)


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 87.7MB/s]


Trainable: 24,577,640 / 24,577,640 params (100.0%)
Output classes: 40


NameError: name 'wandb' is not defined

In [13]:
def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [14]:
def train_one_epoch(model, loader, optimizer, criterion, device, mixup_alpha=0.4):
    model.train()
    total_loss = 0.0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        images, labels_a, labels_b, lam = mixup_data(images, labels, alpha=mixup_alpha)

        optimizer.zero_grad()

        outputs = model(images)

        loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [15]:
def validate_detailed(model, loader, criterion, device, num_classes, return_confusion=False, label_names=None):
    model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating", leave=False):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    all_probs = np.concatenate(all_probs)

    accuracy = (all_preds == all_labels).mean()

    precision_macro = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall_macro = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    precision_weighted = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall_weighted = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1_weighted = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

    labels_onehot = np.zeros((len(all_labels), num_classes), dtype=np.int32)
    labels_onehot[np.arange(len(all_labels)), all_labels] = 1

    per_class_ap = {}
    for i in range(num_classes):
        if labels_onehot[:, i].sum() > 0:
            per_class_ap[i] = average_precision_score(labels_onehot[:, i], all_probs[:, i])
        else:
            per_class_ap[i] = float('nan')

    valid_aps = [v for v in per_class_ap.values() if not np.isnan(v)]
    mAP = float(np.mean(valid_aps)) if valid_aps else 0.0


    valid_classes = [(k, v) for k, v in per_class_ap.items() if not np.isnan(v)]
    if valid_classes:
        worst_class, worst_ap = min(valid_classes, key=lambda x: x[1])
        best_class, best_ap = max(valid_classes, key=lambda x: x[1])

    results = {
          'loss': total_loss / len(loader),
          'accuracy': accuracy,
          'mAP': mAP,
          'f1_macro': f1_macro,
          'precision_macro': precision_macro,
          'recall_macro': recall_macro,
          'precision_weighted': precision_weighted,
          'recall_weighted': recall_weighted,
          'f1_weighted': f1_weighted,
          'per_class_ap': per_class_ap,
          'predictions': all_preds,
          'labels': all_labels,
      }

    if return_confusion:
        results['confusion_matrix'] = confusion_matrix(all_labels, all_preds)

    model.train()
    return results

In [16]:

def compute_map(model, loader, device, num_classes, return_per_class=False):

    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Computing mAP", leave=False):
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)

            all_preds.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    preds = np.vstack(all_preds)
    labels = np.hstack(all_labels)

    labels_onehot = np.zeros((len(labels), num_classes), dtype=np.int32)
    labels_onehot[np.arange(len(labels)), labels] = 1

    aps = {}
    for i in range(num_classes):
        if labels_onehot[:, i].sum() > 0:
            aps[i] = average_precision_score(labels_onehot[:, i], preds[:, i])
        else:
            aps[i] = float('nan')

    valid_aps = [v for v in aps.values() if not np.isnan(v)]
    mean_ap = float(np.mean(valid_aps)) if valid_aps else 0.0

    print(f" mAP: {mean_ap:.4f}")

    sorted_aps = sorted([(k, v) for k, v in aps.items() if not np.isnan(v)], key=lambda x: x[1])
    print(f"Worst class (AP {sorted_aps[0][1]:.3f}): {sorted_aps[0][0]}")
    print(f"Best class (AP {sorted_aps[-1][1]:.3f}): {sorted_aps[-1][0]}")

    model.train()

    if return_per_class:
        return mean_ap, aps
    return mean_ap

In [23]:
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = CosineAnnealingLR(
    optimizer,
    T_max=10
)

epochs = 10

for epoch in range(epochs):

    train_loss = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )

    metrics = validate_detailed(
        model, val_loader, criterion, device, num_classes
    )

    scheduler.step()

    wandb.log({
        "stage": "frozen",
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": metrics['loss'],
        "val_accuracy": metrics['accuracy'],
        "val_mAP": metrics['mAP'],
        "val_f1": metrics['f1_macro'],
        "learning_rate": optimizer.param_groups[0]['lr'],
    })

    print(f"\n{'─'*50}")
    print(f"[Frozen] Epoch {epoch + 1}/{epochs}")
    print(f"  LR:          {optimizer.param_groups[0]['lr']:.2e}")
    print(f"  Train Loss:  {train_loss:.4f}")
    print(f"  Val Loss:    {metrics['loss']:.4f}")
    print(f"  mAP:         {metrics['mAP']:.4f}   ")
    print(f"  F1 Macro:    {metrics['f1_macro']:.4f}   ")
    print(f"  F1 Weighted: {metrics['f1_weighted']:.4f}   ")
    print(f"{'─'*50}")


──────────────────────────────────────────────────
[Frozen] Epoch 1/10
  LR:          9.76e-04
  Train Loss:  2.7978
  Val Loss:    1.9829
  mAP:         0.4761   
  F1 Macro:    0.3728   
  F1 Weighted: 0.3775   
──────────────────────────────────────────────────



──────────────────────────────────────────────────
[Frozen] Epoch 2/10
  LR:          9.05e-04
  Train Loss:  2.0837
  Val Loss:    1.4653
  mAP:         0.5705   
  F1 Macro:    0.4688   
  F1 Weighted: 0.4684   
──────────────────────────────────────────────────



──────────────────────────────────────────────────
[Frozen] Epoch 3/10
  LR:          7.94e-04
  Train Loss:  1.9539
  Val Loss:    1.4777
  mAP:         0.5878   
  F1 Macro:    0.4903   
  F1 Weighted: 0.4893   
──────────────────────────────────────────────────



──────────────────────────────────────────────────
[Frozen] Epoch 4/10
  LR:          6.55e-04
  Train Loss:  1.8379
  Val Loss:    1.2348
  mAP:         0.6225   
  F1 Macro:    0.5219   
  F1 Weighted: 0.5200   
──────────────────────────────────────────────────



──────────────────────────────────────────────────
[Frozen] Epoch 5/10
  LR:          5.00e-04
  Train Loss:  1.6910
  Val Loss:    1.1919
  mAP:         0.6366   
  F1 Macro:    0.5264   
  F1 Weighted: 0.5246   
──────────────────────────────────────────────────



──────────────────────────────────────────────────
[Frozen] Epoch 6/10
  LR:          3.45e-04
  Train Loss:  1.6953
  Val Loss:    1.0703
  mAP:         0.6646   
  F1 Macro:    0.5715   
  F1 Weighted: 0.5724   
──────────────────────────────────────────────────



──────────────────────────────────────────────────
[Frozen] Epoch 7/10
  LR:          2.06e-04
  Train Loss:  1.6473
  Val Loss:    1.1261
  mAP:         0.6597   
  F1 Macro:    0.5902   
  F1 Weighted: 0.5892   
──────────────────────────────────────────────────



──────────────────────────────────────────────────
[Frozen] Epoch 8/10
  LR:          9.55e-05
  Train Loss:  1.6088
  Val Loss:    1.1045
  mAP:         0.6657   
  F1 Macro:    0.5876   
  F1 Weighted: 0.5866   
──────────────────────────────────────────────────



──────────────────────────────────────────────────
[Frozen] Epoch 9/10
  LR:          2.45e-05
  Train Loss:  1.5175
  Val Loss:    1.0194
  mAP:         0.6827   
  F1 Macro:    0.6007   
  F1 Weighted: 0.5998   
──────────────────────────────────────────────────



──────────────────────────────────────────────────
[Frozen] Epoch 10/10
  LR:          0.00e+00
  Train Loss:  1.5657
  Val Loss:    1.0196
  mAP:         0.6805   
  F1 Macro:    0.6034   
  F1 Weighted: 0.6026   
──────────────────────────────────────────────────


In [24]:
for param in model.parameters():
    param.requires_grad = True

EPOCHS = 30
FINE_TUNE_LR = 1e-4
EARLY_STOP = 7
MIXUP_ALPHA = 0.2

optimizer = optim.Adam(
    model.parameters(),
    lr=FINE_TUNE_LR,
    weight_decay=1e-4
)

scheduler = CosineAnnealingLR(
    optimizer,
    T_max = EPOCHS,
    eta_min = 1e-6
)

best_map = 0.0
best_metrics = {}
epochs_no_improve = 0

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(
        model, train_loader, optimizer, criterion, device,
        mixup_alpha=MIXUP_ALPHA
    )


    metrics = validate_detailed(
        model, val_loader, criterion, device,
        num_classes=num_classes,
        return_confusion=True,
        label_names={v: k for k, v in label_map.items()}
    )

    scheduler.step()

    cm = metrics['confusion_matrix']
    per_class_acc = cm.diagonal() / (cm.sum(axis=1) + 1e-8)


    print(f"\n{'─'*55}")
    print(f"[Fine-tune] Epoch {epoch+1}/{EPOCHS}")
    print(f"  LR:            {optimizer.param_groups[0]['lr']:.2e}")
    print(f"  Train Loss:    {train_loss:.4f}")
    print(f"  Val Loss:      {metrics['loss']:.4f}")
    print(f"  mAP:           {metrics['mAP']:.4f}   ")
    print(f"  F1 Macro:      {metrics['f1_macro']:.4f}  ")
    print(f"  F1 Weighted:   {metrics['f1_weighted']:.4f} ")
    print(f"  Precision:     {metrics['precision_macro']:.4f}")
    print(f"  Recall:        {metrics['recall_macro']:.4f}")
    print(f"  Mean cls acc:  {per_class_acc.mean():.4f}")
    print(f"  Worst cls acc: {per_class_acc.min():.4f} "
          f"(class {per_class_acc.argmin()})")

    if 'per_class_ap' in metrics:
        worst = sorted(metrics['per_class_ap'].items(), key=lambda x: x[1])[:3]
        print(f"  Worst 3 AP:    {[(c, f'{ap:.2f}') for c, ap in worst]}")

    print(f"{'─'*55}")

    wandb.log({
        "stage":           "finetune",
        "epoch":           epoch + 1,
        "learning_rate":   optimizer.param_groups[0]['lr'],
        "train_loss":      train_loss,
        "val_loss":        metrics['loss'],
        "val_mAP":         metrics['mAP'],
        "val_f1_macro":    metrics['f1_macro'],
        "val_f1_weighted": metrics['f1_weighted'],
        "val_precision":   metrics['precision_macro'],
        "val_recall":      metrics['recall_macro'],
        "mean_class_acc":  per_class_acc.mean(),
        "worst_class_acc": per_class_acc.min(),
    })


    if metrics['mAP'] > best_map:
        best_map     = metrics['mAP']
        best_metrics = {
            'mAP':       metrics['mAP'],
            'f1_macro':  metrics['f1_macro'],
            'precision': metrics['precision_macro'],
            'recall':    metrics['recall_macro'],
        }
        torch.save({
            'model_state_dict': model.state_dict(),
            'label_map':        label_map,
            'num_classes':      num_classes,
            'architecture':     'resnet50',
        }, SAVE_PATH)

        print(f"  Saved best model (mAP: {best_map:.4f}) → {SAVE_PATH}")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"  No improvement ({epochs_no_improve}/{EARLY_STOP})")
        if epochs_no_improve >= EARLY_STOP:
            print(" Early stopping triggered")
            break

print(" BEST RESULTS (Fine-tune Stage):")

for k, v in best_metrics.items():
    print(f"  {k:<12}: {v:.4f}")




───────────────────────────────────────────────────────
[Fine-tune] Epoch 1/30
  LR:            9.97e-05
  Train Loss:    1.1503
  Val Loss:      0.6786
  mAP:           0.7786   
  F1 Macro:      0.7084  
  F1 Weighted:   0.7061 
  Precision:     0.7390
  Recall:        0.7025
  Mean cls acc:  0.7025
  Worst cls acc: 0.3000 (class 31)
  Worst 3 AP:    [(12, '0.33'), (31, '0.34'), (2, '0.37')]
───────────────────────────────────────────────────────
  Saved best model (mAP: 0.7786) → /content/drive/MyDrive/disease/best_model2.pth



───────────────────────────────────────────────────────
[Fine-tune] Epoch 2/30
  LR:            9.89e-05
  Train Loss:    0.9198
  Val Loss:      0.6299
  mAP:           0.7930   
  F1 Macro:      0.7272  
  F1 Weighted:   0.7251 
  Precision:     0.7581
  Recall:        0.7250
  Mean cls acc:  0.7250
  Worst cls acc: 0.2000 (class 31)
  Worst 3 AP:    [(2, '0.31'), (11, '0.39'), (12, '0.41')]
───────────────────────────────────────────────────────
  Saved best model (mAP: 0.7930) → /content/drive/MyDrive/disease/best_model2.pth



───────────────────────────────────────────────────────
[Fine-tune] Epoch 3/30
  LR:            9.76e-05
  Train Loss:    0.7747
  Val Loss:      0.5356
  mAP:           0.8285   
  F1 Macro:      0.7712  
  F1 Weighted:   0.7697 
  Precision:     0.8077
  Recall:        0.7700
  Mean cls acc:  0.7700
  Worst cls acc: 0.3000 (class 31)
  Worst 3 AP:    [(2, '0.44'), (12, '0.46'), (31, '0.52')]
───────────────────────────────────────────────────────
  Saved best model (mAP: 0.8285) → /content/drive/MyDrive/disease/best_model2.pth



───────────────────────────────────────────────────────
[Fine-tune] Epoch 4/30
  LR:            9.57e-05
  Train Loss:    0.8841
  Val Loss:      0.5302
  mAP:           0.8147   
  F1 Macro:      0.7377  
  F1 Weighted:   0.7358 
  Precision:     0.7591
  Recall:        0.7400
  Mean cls acc:  0.7400
  Worst cls acc: 0.1000 (class 33)
  Worst 3 AP:    [(2, '0.34'), (12, '0.40'), (34, '0.51')]
───────────────────────────────────────────────────────
  No improvement (1/7)



───────────────────────────────────────────────────────
[Fine-tune] Epoch 5/30
  LR:            9.34e-05
  Train Loss:    0.7684
  Val Loss:      0.5482
  mAP:           0.8189   
  F1 Macro:      0.7415  
  F1 Weighted:   0.7407 
  Precision:     0.7780
  Recall:        0.7400
  Mean cls acc:  0.7400
  Worst cls acc: 0.2000 (class 28)
  Worst 3 AP:    [(12, '0.32'), (31, '0.47'), (2, '0.48')]
───────────────────────────────────────────────────────
  No improvement (2/7)


...


───────────────────────────────────────────────────────
[Fine-tune] Epoch 25/30
  LR:            7.63e-06
  Train Loss:    0.5453
  Val Loss:      0.4811
  mAP:           0.8581   
  F1 Macro:      0.7804  
  F1 Weighted:   0.7790 
  Precision:     0.8024
  Recall:        0.7825
  Mean cls acc:  0.7825
  Worst cls acc: 0.2000 (class 31)
  Worst 3 AP:    [(33, '0.39'), (31, '0.45'), (2, '0.57')]
───────────────────────────────────────────────────────
  No improvement (3/7)



───────────────────────────────────────────────────────
[Fine-tune] Epoch 26/30
  LR:            5.28e-06
  Train Loss:    0.5590
  Val Loss:      0.4831
  mAP:           0.8583   
  F1 Macro:      0.7735  
  F1 Weighted:   0.7720 
  Precision:     0.7909
  Recall:        0.7775
  Mean cls acc:  0.7775
  Worst cls acc: 0.2000 (class 31)
  Worst 3 AP:    [(33, '0.39'), (31, '0.45'), (2, '0.55')]
───────────────────────────────────────────────────────
  No improvement (4/7)



───────────────────────────────────────────────────────
[Fine-tune] Epoch 27/30
  LR:            3.42e-06
  Train Loss:    0.6359
  Val Loss:      0.5031
  mAP:           0.8552   
  F1 Macro:      0.7779  
  F1 Weighted:   0.7765 
  Precision:     0.7977
  Recall:        0.7800
  Mean cls acc:  0.7800
  Worst cls acc: 0.2000 (class 31)
  Worst 3 AP:    [(33, '0.39'), (31, '0.44'), (2, '0.52')]
───────────────────────────────────────────────────────
  No improvement (5/7)



───────────────────────────────────────────────────────
[Fine-tune] Epoch 28/30
  LR:            2.08e-06
  Train Loss:    0.5846
  Val Loss:      0.4884
  mAP:           0.8564   
  F1 Macro:      0.7800  
  F1 Weighted:   0.7786 
  Precision:     0.7983
  Recall:        0.7825
  Mean cls acc:  0.7825
  Worst cls acc: 0.2000 (class 31)
  Worst 3 AP:    [(33, '0.39'), (31, '0.45'), (2, '0.55')]
───────────────────────────────────────────────────────
  No improvement (6/7)



───────────────────────────────────────────────────────
[Fine-tune] Epoch 29/30
  LR:            1.27e-06
  Train Loss:    0.4473
  Val Loss:      0.4975
  mAP:           0.8598   
  F1 Macro:      0.7788  
  F1 Weighted:   0.7774 
  Precision:     0.7978
  Recall:        0.7825
  Mean cls acc:  0.7825
  Worst cls acc: 0.2000 (class 31)
  Worst 3 AP:    [(33, '0.38'), (31, '0.44'), (1, '0.59')]
───────────────────────────────────────────────────────
  No improvement (7/7)
 Early stopping triggered
 BEST RESULTS (Fine-tune Stage):
  mAP         : 0.8645
  f1_macro    : 0.7889
  precision   : 0.8085
  recall      : 0.7900


In [25]:
cm = metrics['confusion_matrix']
classes = [33, 31, 2]
for i in classes:
    print(f"Class {i} predictions: {cm[i][classes]}")

Class 33 predictions: [2 6 0]
Class 31 predictions: [3 2 0]
Class 2 predictions: [0 0 8]


In [17]:
print(label_to_name[31])
print(label_to_name[33])
print(label_to_name[2])

bacterial leaf spot
septoria leaf spot
rust


In [23]:
def predict(image_path, model, label_map, device, threshold=0.4):

    label_to_name = {v: k for k, v in label_map.items()}

    transform = transforms.Compose([
      transforms.Resize((256, 256)),
      transforms.CenterCrop(224),
      transforms.ToTensor(),
      transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
  ])

    try:
        img = Image.open(image_path).convert("RGB")
    except Exception as e:
        return {"error": f"Cannot open image: {e}", "reliable": False}

    try:
        img_tensor = transform(img).unsqueeze(0).to(device)
    except Exception as e:
        return {"error": f"Transform failed: {e}", "reliable": False}
    try:
        model.eval()
        with torch.no_grad():
            outputs    = model(img_tensor)
            probs      = torch.softmax(outputs, dim=1)
            confidence, predicted = probs.max(dim=1)
            confidence = confidence.item()
            predicted  = predicted.item()
    except Exception as e:
        return {"error": f"Prediction failed: {e}", "reliable": False}

    if confidence < threshold:
        return {
            "disease": "Uncertain — please try a clearer photo",,
            "confidence": round(confidence, 4),
            "reliable":   False
        }

    return {
        "disease":    label_to_name[predicted],
        "confidence": round(confidence, 4),
        "reliable":   True
    }

In [ ]:
print(torch.cuda.is_available())